# 1M Row Scaling Benchmark (2xT4 Multi-GPU)

Tests subsample annealing, mini-batch Gibbs, parallel vs sequential row scoring at 1M scale.
Uses `jax.pmap` for multi-chain GPU distribution.

- **Min VRAM**: 16GB per device
- **Designed for**: Kaggle 2xT4 (32GB total VRAM)
- **Kernels tested**: `packed_transition_row_assignments_parallel`, `packed_transition_row_assignments_minibatch`, `minibatch_gibbs_sweep`, `subsample_anneal`

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/kaggle/working/jaxcross"
BRANCH = "main"  # @param {type:"string"}

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)

!git fetch origin && (git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}) && git pull origin {BRANCH}

%pip install -e . --no-deps -q

print(f"Branch: {BRANCH}")
print(f"Working directory: {os.getcwd()}")
print("Setup complete.")

In [ ]:
import gc
import json
import shutil
import time

import jax
import jax.numpy as jnp
import numpy as np

import crosscat.packed.state as _ps
from benchmarks.utils import detect_platform, make_benchmark_data
from crosscat import (
    initialize,
    minibatch_gibbs_sweep,
    pack_state,
    packed_gibbs_sweep,
    packed_insert_rows,
    packed_transition_row_assignments_minibatch,
    packed_transition_row_assignments_parallel,
    subsample_anneal,
    suggest_max_clusters,
)
from crosscat.packed import batch_packed_states
from crosscat.packed.kernels import packed_log_joint

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
n_devices = jax.device_count()
print(f"JAX devices: {n_devices} ({jax.devices()})")
print(f"suggest_max_clusters(1_000_000) = {suggest_max_clusters(1_000_000)}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Parallel vs Sequential Row Scoring (100K rows)

Compare `packed_transition_row_assignments_parallel` vs `_minibatch` on 100K rows.

In [ ]:
print("--- Parallel vs Sequential: 100,000 rows x 20 cols ---")
k1, k2, k3, k4 = jax.random.split(jax.random.key(42), 4)
data_100k, col_types_100k = make_benchmark_data(k1, 100_000, 20)
max_k = suggest_max_clusters(100_000)

state = initialize(k2, data_100k, col_types_100k).state
packed = pack_state(state, max_clusters=max_k)
packed = packed_gibbs_sweep(k3, packed, data_100k, n_sweeps=3)
packed.column_assignments.block_until_ready()

t0 = time.perf_counter()
p_packed = packed_transition_row_assignments_parallel(k4, packed, data_100k)
p_packed.column_assignments.block_until_ready()
parallel_time = time.perf_counter() - t0
print(f"  parallel row sweep: {parallel_time:.2f}s")

k5 = jax.random.fold_in(k4, 1)
t0 = time.perf_counter()
m_packed = packed_transition_row_assignments_minibatch(k5, packed, data_100k, batch_size=10_000)
m_packed.column_assignments.block_until_ready()
minibatch_time = time.perf_counter() - t0
print(f"  mini-batch (10K) row sweep: {minibatch_time:.2f}s")
print(f"  parallel speedup vs mini-batch: {minibatch_time / max(parallel_time, 0.001):.1f}x")

del data_100k, packed, p_packed, m_packed, state
gc.collect()

## 3. Define pmap Sweep Function

In [ ]:
def _sweep_one_chain(key, packed, data, n_sweeps):
    """Run n_sweeps of packed Gibbs on a single chain."""
    return packed_gibbs_sweep(key, packed, data, n_sweeps=n_sweeps)


def _sweep_chains_on_device(keys, packed_batch, data, n_sweeps):
    """Run sweeps for chains_per_device chains on one device.

    keys: (chains_per_device,) array of PRNG keys
    packed_batch: batched PackedCrossCatState with leading dim
    """

    def body(i, carry):
        packed_b = carry
        single_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)[i]
        for name in _ps._STATIC_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)
        single = _ps.PackedCrossCatState(**single_kwargs)

        result = _sweep_one_chain(keys[i], single, data, n_sweeps)

        new_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            arr = getattr(packed_b, name)
            new_kwargs[name] = arr.at[i].set(getattr(result, name))
        for name in _ps._STATIC_FIELDS:
            new_kwargs[name] = getattr(packed_b, name)
        return _ps.PackedCrossCatState(**new_kwargs)

    return jax.lax.fori_loop(0, keys.shape[0], body, packed_batch)


pmap_sweep = jax.pmap(
    _sweep_chains_on_device,
    in_axes=(0, 0, None, None),
    static_broadcasted_argnums=(3,),
)
print("pmap sweep function defined")

## 4. Multi-Chain Mini-batch Throughput (1M rows, pmap)

Init on 5K subsample, batch insert to 1M, then run mini-batch Gibbs sweeps
with 2 chains distributed across GPUs.

In [ ]:
N_CHAINS = max(2, n_devices)
CHAINS_PER_DEVICE = N_CHAINS // n_devices
N_CHAINS = CHAINS_PER_DEVICE * n_devices
assert CHAINS_PER_DEVICE >= 1, (
    f"Need at least 1 chain per device: N_CHAINS must be >= n_devices ({n_devices})"
)

print(f"--- Mini-batch Throughput: 1M rows, {N_CHAINS} chains ({CHAINS_PER_DEVICE}/device) ---")
k1, k2, k3 = jax.random.split(jax.random.key(43), 3)
data_1m, col_types_1m = make_benchmark_data(k1, 1_000_000, 20)
max_k = suggest_max_clusters(1_000_000)
print(f"Data: {data_1m.nbytes / (1024**2):.0f} MB, max_clusters: {max_k}")

# Init chains on subsample
init_keys = jax.random.split(k2, N_CHAINS)
all_packed = []
sub_idx = None
for c in range(N_CHAINS):
    result = initialize(init_keys[c], data_1m, col_types_1m, subsample_rows=5000)
    all_packed.append(pack_state(result.state, max_clusters=max_k))
    if sub_idx is None:
        sub_idx = result.subsample_idx
sub_data = data_1m[sub_idx]
print(f"Initialized {N_CHAINS} chains on 5K subsample")

# Pre-sweeps via pmap
sweep_keys = jax.random.split(jax.random.key(44), N_CHAINS)
batched = batch_packed_states(all_packed)
keys_pmap = sweep_keys.reshape(n_devices, CHAINS_PER_DEVICE, *sweep_keys.shape[1:])
batched_pmap_kwargs = {}
for name in _ps._ARRAY_FIELDS:
    arr = getattr(batched, name)
    batched_pmap_kwargs[name] = arr.reshape((n_devices, CHAINS_PER_DEVICE) + arr.shape[1:])
for name in _ps._STATIC_FIELDS:
    batched_pmap_kwargs[name] = getattr(batched, name)
batched_pmap = _ps.PackedCrossCatState(**batched_pmap_kwargs)

t0 = time.perf_counter()
result_pmap = pmap_sweep(keys_pmap, batched_pmap, sub_data, 3)
jax.tree.map(lambda x: x.block_until_ready(), result_pmap)
print(f"Pre-sweeps: {time.perf_counter() - t0:.2f}s (includes JIT)")

# Unflatten
for c in range(N_CHAINS):
    dev_idx = c // CHAINS_PER_DEVICE
    chain_idx = c % CHAINS_PER_DEVICE
    kwargs = {}
    for name in _ps._ARRAY_FIELDS:
        kwargs[name] = getattr(result_pmap, name)[dev_idx][chain_idx]
    for name in _ps._STATIC_FIELDS:
        kwargs[name] = getattr(result_pmap, name)
    all_packed[c] = _ps.PackedCrossCatState(**kwargs)

# Batch insert remaining rows
included = jnp.zeros(1_000_000, dtype=bool).at[sub_idx].set(True)
remaining_idx = jnp.where(~included, size=1_000_000 - 5000)[0]
remaining = data_1m[remaining_idx]
batch_size = 50_000
current_data = sub_data
t0 = time.perf_counter()
for b in range(0, remaining.shape[0], batch_size):
    batch = remaining[b : b + batch_size]
    for c in range(N_CHAINS):
        kb = jax.random.fold_in(jax.random.key(45 + c), b)
        all_packed[c], _ = packed_insert_rows(kb, all_packed[c], current_data, batch)
    current_data = jnp.concatenate([current_data, batch], axis=0)
    print(f"  Inserted to {all_packed[0].n_rows:,} rows, {time.perf_counter() - t0:.1f}s")
full_data = current_data
print(f"Insert complete: {time.perf_counter() - t0:.1f}s")

# Mini-batch sweeps (sequential per chain)
n_mb_sweeps = 5
t0 = time.perf_counter()
for c in range(N_CHAINS):
    kc = jax.random.fold_in(k3, c)
    all_packed[c] = minibatch_gibbs_sweep(
        kc,
        all_packed[c],
        full_data,
        batch_size=10_000,
        n_sweeps=n_mb_sweeps,
    )
    all_packed[c].column_assignments.block_until_ready()
    print(f"  Chain {c + 1}: {n_mb_sweeps} mini-batch sweeps done")
mb_total = time.perf_counter() - t0
mb_per_sweep = mb_total / (n_mb_sweeps * N_CHAINS)
print(f"Mini-batch total: {mb_total:.1f}s ({mb_per_sweep:.1f}s per chain-sweep)")

# Score chains
scores = [float(packed_log_joint(p, full_data)) for p in all_packed]
best = int(np.argmax(scores))
print(f"\nBest chain: {best + 1} (log_joint={scores[best]:.0f})")

## 5. Subsample Annealing (2K -> 1M)

`subsample_anneal()` grows the dataset progressively. Single chain --
tests the annealing workflow at scale.

In [ ]:
print("--- Subsample Annealing: 1,000,000 rows x 20 cols ---")
k_anneal = jax.random.fold_in(jax.random.key(46), 3)
data_anneal, col_types_anneal = make_benchmark_data(k_anneal, 1_000_000, 20)
print(f"Data: {data_anneal.nbytes / (1024**2):.0f} MB")

t0 = time.perf_counter()
packed_annealed, reordered_data = subsample_anneal(
    k_anneal,
    data_anneal,
    col_types_anneal,
    initial_size=2000,
    growth_factor=4.0,
    sweeps_per_stage=5,
)
anneal_time = time.perf_counter() - t0
print(f"Annealing complete: {packed_annealed.n_rows:,} rows, {anneal_time:.1f}s")

## 6. Summary

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Devices: {n_devices}x {platform['gpu_names']}")
print(f"  Parallel row sweep (100K): {parallel_time:.2f}s")
print(f"  Mini-batch per chain-sweep (1M): {mb_per_sweep:.1f}s")
print(f"  Subsample annealing to 1M: {anneal_time:.1f}s")
print("=" * 60)

from pathlib import Path

results_dir = Path("benchmarks/results/scaling")
results_dir.mkdir(parents=True, exist_ok=True)

results = {
    "backend": platform["backend"],
    "devices": [str(d) for d in jax.devices()],
    "n_devices": n_devices,
    "parallel_time": parallel_time,
    "minibatch_time": minibatch_time,
    "mb_total": mb_total,
    "mb_per_sweep": mb_per_sweep,
    "anneal_time": anneal_time,
    "anneal_final_rows": int(packed_annealed.n_rows),
    "best_chain_scores": scores,
}
with open(results_dir / "scaling_1m_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {results_dir / 'scaling_1m_results.json'}")

# Archive to /kaggle/working/ for easy download

results_tar = "/kaggle/working/scaling_1m_results.tar.gz"
shutil.make_archive("/kaggle/working/scaling_1m_results", "gztar", ".", str(results_dir))
print(f"Archived to {results_tar}")
print("Download from Kaggle Output tab.")

for f in sorted(results_dir.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        print(f"  {f.relative_to(results_dir)}  ({size:,} bytes)")